LAB5 ขั้นเตรียมการ: สร้างชุดข้อมูล CSV สำหรับใบงานที่ 5 (ครูรันครั้งเดียว)
ผลลัพธ์: data/iris.csv, data/wine.csv, data/customers.csv

In [2]:
%pip install pandas scikit-learn numpy matplotlib seaborn


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# นำเข้า pathlib เพื่อใช้ในการจัดการเส้นทางของไฟล์และโฟลเดอร์
from pathlib import Path

In [4]:
# นำเข้า numpy สำหรับการจัดการข้อมูลเชิงตัวเลข
import numpy as np

# pandas สำหรับการจัดการข้อมูลในรูปแบบ DataFrame
import pandas as pd
# นำเข้า datasets จาก sklearn เพื่อโหลดชุดข้อมูลมาตรฐาน
from sklearn.datasets import load_iris, load_wine

In [6]:
# กำหนดเส้นทางฐานของโปรเจกต์ โดยใช้ pathlib เพื่อให้สามารถเข้าถึงไฟล์และโฟลเดอร์ได้อย่างสะดวก
BASE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
# กำหนดเส้นทางของโฟลเดอร์ data โดยใช้ BASE เป็นฐาน
DATA = BASE / "data"
# สร้างโฟลเดอร์ data หากยังไม่มี โดยใช้ pathlib ซึ่งจะไม่เกิดข้อผิดพลาดหากโฟลเดอร์มีอยู่แล้ว
DATA.mkdir(exist_ok=True)

1) Iris Dataset (150 แถว 4 คุณลักษณะ 3 คลาส)

In [7]:
# โหลดชุดข้อมูล iris จาก sklearn
iris = load_iris()
# สร้าง DataFrame จากข้อมูล iris โดยกำหนดชื่อคอลัมน์และเพิ่มคอลัมน์ species ที่แสดงชื่อชนิดของดอกไม้
df = pd.DataFrame(iris.data,
                  columns=["sepal_length", "sepal_width", "petal_length", "petal_width"])
# เพิ่มคอลัมน์ species โดยใช้ชื่อชนิดของดอกไม้จาก iris.target_names
df["species"] = [iris.target_names[t] for t in iris.target]
# บันทึก DataFrame เป็นไฟล์ CSV ในโฟลเดอร์ data โดยไม่รวมดัชนีแถว
df.to_csv(DATA / "iris.csv", index=False)
# แสดงขนาดของ DataFrame ที่สร้างขึ้น
print("สร้าง data/iris.csv      :", df.shape)

สร้าง data/iris.csv      : (150, 5)


2) Wine Dataset (178 แถว 13 คุณลักษณะ 3 คลาส) — สำหรับงานท้าทายเพิ่มเติม

In [8]:
# 2) Wine Dataset (178 แถว 13 คุณลักษณะ 3 คลาส) — สำหรับงานท้าทายเพิ่มเติม
wine = load_wine()
# สร้าง DataFrame จากข้อมูล wine โดยกำหนดชื่อคอลัมน์และเพิ่มคอลัมน์ wine_class ที่แสดงชนิดของไวน์
dfw = pd.DataFrame(wine.data, columns=wine.feature_names)
# เพิ่มคอลัมน์ wine_class โดยใช้ค่า target จากชุดข้อมูล wine
dfw["wine_class"] = wine.target
# บันทึก DataFrame เป็นไฟล์ CSV ในโฟลเดอร์ data โดยไม่รวมดัชนีแถว
dfw.to_csv(DATA / "wine.csv", index=False)
# แสดงขนาดของ DataFrame ที่สร้างขึ้น
print("สร้าง data/wine.csv      :", dfw.shape)

สร้าง data/wine.csv      : (178, 14)


3) Customer Segmentation (ข้อมูลจำลอง 200 แถว สำหรับแบบฝึกหัด K-Means)

In [11]:
# 3) Customer Segmentation (ข้อมูลจำลอง 200 แถว สำหรับแบบฝึกหัด K-Means)
rng = np.random.default_rng(42)
# กำหนดกลุ่มลูกค้าแต่ละกลุ่ม โดยระบุจำนวนลูกค้า รายได้เฉลี่ย ค่าใช้จ่ายเฉลี่ย และอายุเฉลี่ย
groups = [
    # (จำนวน, รายได้เฉลี่ย(พันบาท/เดือน), ค่าใช้จ่ายเฉลี่ย(คะแนน 1-100), อายุเฉลี่ย)
    (50, 20, 25, 45),   # รายได้น้อย ใช้จ่ายน้อย
    (50, 22, 75, 24),   # รายได้น้อย ใช้จ่ายมาก (วัยรุ่น)
    (50, 70, 20, 50),   # รายได้มาก ใช้จ่ายน้อย (ประหยัด)
    (50, 75, 80, 33),   # รายได้มาก ใช้จ่ายมาก (พรีเมียม)
]
# สร้าง DataFrame สำหรับลูกค้าแต่ละกลุ่ม โดยสุ่มค่ารายได้ ค่าใช้จ่าย และอายุจากการแจกแจงปกติ (Normal Distribution) และจำกัดค่าให้อยู่ในช่วงที่เหมาะสม
rows = []
for n, inc, spend, age in groups:
    rows.append(pd.DataFrame({
        "income_k": rng.normal(inc, 6, n).round(1),
        "spending_score": np.clip(rng.normal(spend, 9, n), 1, 100).round(0),
        "age": np.clip(rng.normal(age, 6, n), 18, 70).round(0),
    }))
# รวม DataFrame ของลูกค้าทั้งหมดเข้าด้วยกัน สุ่มลำดับแถว และรีเซ็ตดัชนี
dfc = pd.concat(rows, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
# เพิ่มคอลัมน์ customer_id โดยใช้หมายเลขลำดับตั้งแต่ 1 ถึงจำนวนแถวของ DataFrame
dfc.insert(0, "customer_id", range(1, len(dfc) + 1))
# บันทึก DataFrame เป็นไฟล์ CSV ในโฟลเดอร์ data โดยไม่รวมดัชนีแถว
dfc.to_csv(DATA / "customers.csv", index=False)
# แสดงขนาดของ DataFrame ที่สร้างขึ้น
print("สร้าง data/customers.csv :", dfc.shape)

สร้าง data/customers.csv : (200, 4)


In [10]:

print("เสร็จสิ้น — ข้อมูลทั้งหมดอยู่ในโฟลเดอร์ data/")

เสร็จสิ้น — ข้อมูลทั้งหมดอยู่ในโฟลเดอร์ data/
